# 05 · PPO with a KL penalty on a toy policy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/post-training/05_ppo_toy.ipynb) &nbsp; 📖 Concept deck: [RLHF with PPO](https://nilesh-salpe.github.io/visual-ai/post-training/05-rlhf-ppo/) &nbsp; ⏱️ Suggested time: **45 min**

**You will build:** the KL-penalized reward used in RLHF, the PPO clipped objective, and a training loop on a one-step "language model" whose exact optimum you can compute and compare against.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def log_softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=axis, keepdims=True))

def log_sigmoid(z):
    return -np.logaddexp(0, -z)

V = 8                                             # a "vocabulary" of 8 possible responses
theta_ref = np.array([2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5])   # SFT / reference policy logits
reward = np.array([0.0, 0.2, 0.5, 0.1, 1.0, 0.3, 2.0, 0.0])        # reward model score per response
pi_ref = softmax(theta_ref)

def optimal_policy(beta):
    """Closed-form optimum of E[r] - beta*KL(pi || pi_ref): pi* ∝ pi_ref * exp(r / beta)."""
    return softmax(np.log(pi_ref) + reward / beta)

def kl(p, q):
    return float((p * (np.log(p) - np.log(q))).sum())

## Task 1: The KL-penalized reward

InstructGPT adds *"a per-token KL penalty from the SFT model at each token to mitigate overoptimization of the reward model"*:
$$R(x, y) = r_\theta(x,y) - \beta \log\frac{\pi^{RL}(y\mid x)}{\pi^{SFT}(y\mid x)}$$
Given per-sample rewards, policy log-probs and reference log-probs, return R. Averaged over samples from π, the penalty term equals β·KL(π‖π_ref).

In [ ]:
def kl_penalized_reward(r, logp, logp_ref, beta):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_kl_penalized_reward(f):
    assert np.allclose(f(np.array([1.0, 2.0]), np.log([.5, .25]), np.log([.25, .25]), 0.5), [1 - 0.5 * np.log(2), 2.0])
    p = softmax(np.array([1.0, 0, -1])); q = softmax(np.zeros(3)); r_ = np.zeros(3)
    assert np.isclose((p * f(r_, np.log(p), np.log(q), 1.0)).sum(), -kl(p, q)), "expected penalty = beta * KL"

check("kl_penalized_reward", _test_kl_penalized_reward, kl_penalized_reward)

## Task 2: The PPO clipped objective

PPO (Schulman et al. 2017) maximizes
$$L^{CLIP} = \mathbb{E}\big[\min\big(\rho\,\hat A,\ \mathrm{clip}(\rho, 1-\epsilon, 1+\epsilon)\,\hat A\big)\big],\qquad \rho = \frac{\pi_\theta(a)}{\pi_{\theta_{old}}(a)}$$
with ε = 0.2 in the paper. The `min` makes it pessimistic: once the ratio moves past 1±ε in the direction that helps, the gradient stops, so one batch can't push the policy too far.
Return the objective (to **maximize**) given new and old log-probs and advantages.

In [ ]:
def ppo_clip_objective(logp_new, logp_old, adv, eps=0.2):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_ppo_clip_objective(f):
    lo = np.log(np.full(4, .25)); ln = np.log([.25, .5, .1, .4]); A = np.array([1.0, 1.0, -1.0, -1.0])
    ratio = np.exp(ln - lo)   # 1, 2, 0.4, 1.6
    want = np.mean([1.0, 1.2, -0.8, -1.6])
    assert np.isclose(f(ln, lo, A), want), f"got {f(ln, lo, A)}, expected {want} (min of unclipped and clipped)"
    assert np.isclose(f(lo, lo, A), A.mean()), "ratio 1 -> plain advantage"

check("ppo_clip_objective", _test_ppo_clip_objective, ppo_clip_objective)

## Task 3: Train the toy policy with PPO

Loop for `iters` iterations:
1. Sample `batch` responses from the current policy $\pi_{old} = \mathrm{softmax}(\theta)$.
2. Compute KL-penalized rewards; advantage = reward − batch mean (a simple baseline instead of a value network).
3. Run `epochs` gradient-**ascent** steps on $L^{CLIP}$ w.r.t. θ. For a softmax policy, $\nabla_\theta \log\pi(a) = e_a - \pi$. Each sample whose term is *not* clipped contributes $\hat A\,\rho\,(e_a - \pi)$; clipped samples contribute 0.

Return final θ and a history of (expected reward, KL to reference). The tests compare your result with the exact optimum $\pi^\star \propto \pi_{ref}\,e^{r/\beta}$.

In [ ]:
def train_ppo(beta=0.5, iters=300, batch=256, epochs=4, lr=0.5, eps=0.2, seed=0):
    rng = np.random.default_rng(seed); theta = theta_ref.copy(); hist = []
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_train_ppo(f):
    th, h = f(beta=0.5)
    p = softmax(th)
    assert h[-1][0] > h[0][0] + 0.3, "expected reward should rise substantially"
    assert kl(optimal_policy(0.5), p) < 0.05, f"should approach pi* ∝ pi_ref*exp(r/beta); KL(pi*||pi) = {kl(optimal_policy(0.5), p):.3f}"
    th2, h2 = f(beta=2.0)
    assert h2[-1][1] < h[-1][1], "a larger beta must keep the policy closer to the reference"

check("train_ppo", _test_train_ppo, train_ppo)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: The KL-penalized reward
def kl_penalized_reward_solution(r, logp, logp_ref, beta):
    return r - beta * (logp - logp_ref)

check("kl_penalized_reward (reference)", _test_kl_penalized_reward, kl_penalized_reward_solution)

In [ ]:
# @title Solution for Task 2: The PPO clipped objective
def ppo_clip_objective_solution(logp_new, logp_old, adv, eps=0.2):
    ratio = np.exp(logp_new - logp_old)
    return np.minimum(ratio * adv, np.clip(ratio, 1 - eps, 1 + eps) * adv).mean()

check("ppo_clip_objective (reference)", _test_ppo_clip_objective, ppo_clip_objective_solution)

In [ ]:
# @title Solution for Task 3: Train the toy policy with PPO
def train_ppo_solution(beta=0.5, iters=300, batch=256, epochs=4, lr=0.5, eps=0.2, seed=0):
    rng = np.random.default_rng(seed); theta = theta_ref.copy(); hist = []
    for _ in range(iters):
        p_old = softmax(theta); a = rng.choice(V, size=batch, p=p_old)
        R = kl_penalized_reward_solution(reward[a], np.log(p_old[a]), np.log(pi_ref[a]), beta)
        adv = R - R.mean()
        for _ in range(epochs):
            p = softmax(theta); ratio = p[a] / p_old[a]
            active = ~(((adv > 0) & (ratio > 1 + eps)) | ((adv < 0) & (ratio < 1 - eps)))
            coef = np.where(active, adv * ratio, 0.0)
            grad = (np.bincount(a, weights=coef, minlength=V) - coef.sum() * p) / batch
            theta = theta + lr * grad
        p = softmax(theta); hist.append((float(p @ reward), kl(p, pi_ref)))
    return theta, hist

check("train_ppo (reference)", _test_train_ppo, train_ppo_solution)

---
## Experiments
**β is the leash.** Small β lets the policy chase the reward model; large β keeps it near the SFT model. The exact optimum of $\mathbb{E}[r] - \beta\,\mathrm{KL}$ is $\pi^\star \propto \pi_{ref}e^{r/\beta}$: the same formula DPO builds on (notebook 07).

In [ ]:
print("beta   E[reward]  KL(pi||ref)   P(best response)   | exact optimum E[r], KL")
for beta in [0.1, 0.25, 0.5, 1.0, 2.0, 5.0]:
    th, h = train_ppo_solution(beta=beta, iters=200)
    p, ps = softmax(th), optimal_policy(beta)
    print(f"{beta:4}   {p @ reward:8.3f}   {kl(p, pi_ref):10.3f}   {p[6]:15.2f}     | {ps @ reward:.3f}, {kl(ps, pi_ref):.3f}")

---
## Takeaways
* RLHF maximizes the reward-model score minus β·KL to the SFT model. The KL term stops the policy drifting into regions where the reward model is wrong.
* PPO's clipped ratio limits how far one batch can move the policy, which makes many gradient steps per batch safe.
* Real RLHF keeps four models in memory: the policy, the reference (SFT), the reward model, and a value model (critic) that estimates advantages per token. Here a batch-mean baseline replaces the critic, which is the idea GRPO scales up (notebook 08).